In [2]:
!pip install selenium
!apt-get update
!apt install chromium-chromedriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.7 MB/s eta 0:00:00
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Ign:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [998 kB]
Get:11 https://

In [5]:
import csv
import os
import time
from datetime import date, datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException, ElementClickInterceptedException
import shutil
from google.colab import drive

# Montagem do Google Drive para salvar os dados
drive.mount('/content/drive')

# Definir o caminho para salvar o arquivo CSV na pasta "output" no Google Drive
output_dir = '/content/drive/MyDrive/output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
output_path = os.path.join(output_dir, 'tumelero_produtos.csv')

# Configuração do WebDriver para uso no Colab
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# Inicializa o WebDriver
driver = webdriver.Chrome(options=chrome_options)

# Abre a página do Tumelero
driver.get('https://www.tumelero.com.br/pisos-e-revestimentos/pisos/pisos-ceramicos')

# Espera até que os elementos de produtos estejam visíveis
try:
    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'li[data-smarthintfeature="search"]'))
    )
except TimeoutException:
    print("Tempo de espera expirou. Não foi possível encontrar os produtos.")
    driver.quit()
    exit()

# Função para fechar possíveis overlays de forma segura
def fechar_overlay():
    try:
        overlays = driver.find_elements(By.CSS_SELECTOR, 'div.onetrust-pc-dark-filter')
        for overlay in overlays:
            if overlay.is_displayed():  # Verifica se o overlay está visível
                driver.execute_script("arguments[0].remove();", overlay)
                time.sleep(1)  # Aguarda a remoção do overlay
    except NoSuchElementException:
        print("Nenhum overlay encontrado.")

# Função para tentar clicar no botão "Mais produtos" várias vezes
def clicar_botao_mais_produtos():
    tentativas = 0
    max_tentativas = 3
    while tentativas < max_tentativas:
        try:
            fechar_overlay()  # Tentar fechar qualquer overlay
            botao_mais = WebDriverWait(driver, 30).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.father-loadmore-smarthint button.cf-load-more'))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", botao_mais)
            botao_mais.click()
            time.sleep(2)  # Espera o carregamento dos produtos
            return True
        except (TimeoutException, ElementClickInterceptedException) as e:
            print(f"Erro ao tentar clicar no botão 'Mais produtos': {e}. Tentando novamente ({tentativas + 1}/{max_tentativas})...")
            tentativas += 1
            time.sleep(3)  # Aguarda antes de tentar novamente
    return False

# Função para extrair dados da página
def extrair_dados():
    dados = []
    pagina_atual = 1

    while True:
        try:
            # Coletando os produtos
            produtos = driver.find_elements(By.CSS_SELECTOR, 'li[data-smarthintfeature="search"]')
            print(f"Total de produtos encontrados nesta página: {len(produtos)}")

            for produto in produtos:
                try:
                    nome = produto.find_element(By.CSS_SELECTOR, 'h2.x-shelf__title').text
                    try:
                        preco = produto.find_element(By.CSS_SELECTOR, 'span.x-shelf__best-price strong').text
                    except NoSuchElementException:
                        preco = 'Preço não disponível'
                    imagem = produto.find_element(By.CSS_SELECTOR, 'div.x-shelf__img-container img').get_attribute('src')
                    link = produto.find_element(By.CSS_SELECTOR, 'div.x-shelf__img-container a').get_attribute('href')

                    dados.append({
                        'nome': nome,
                        'preco': preco,
                        'imagem': imagem,
                        'link': link
                    })
                except NoSuchElementException as e:
                    print(f"Elemento não encontrado dentro do produto: {e}")
                    continue

            # Tentar clicar no botão "Mais produtos"
            if not clicar_botao_mais_produtos():
                print("Não foi possível carregar mais produtos. Encerrando coleta.")
                break

            pagina_atual += 1
            print(f"Next page: {pagina_atual}")

        except NoSuchElementException as e:
            print(f"Erro ao encontrar produtos: {e}")
            break

    return dados

# Execução principal
dados = extrair_dados()

# Verifica se dados foram coletados
if not dados:
    print("Nenhum dado coletado.")
else:
    # Exibe os dados coletados no console
    for dado in dados:
        print(f"Nome: {dado['nome']}, Preço: {dado['preco']}, Imagem: {dado['imagem']}, Link: {dado['link']}")

# Salva os dados no arquivo CSV
if dados:
    with open(output_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['nome', 'preco', 'imagem', 'link'])
        writer.writeheader()
        writer.writerows(dados)
    print(f"Dados salvos em {output_path}")

# Fechando o navegador
driver.quit()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total de produtos encontrados nesta página: 36
Erro ao tentar clicar no botão 'Mais produtos': Message: element click intercepted: Element <button class="cf-load-more loadmore-smarthint" nextpage="2" type="button" title="...">Mais produtos</button> is not clickable at point (472, 21). Other element would receive the click: <input class="x-search__input js--search-input" type="search" id="xlow-smarthint-search-input" name="terms" autocomplete="off" placeholder="o que você está procurando?" list="list-options">
  (Session info: chrome=128.0.6613.119)
Stacktrace:
#0 0x5685c563b86a <unknown>
#1 0x5685c5309e50 <unknown>
#2 0x5685c5360346 <unknown>
#3 0x5685c535e25d <unknown>
#4 0x5685c535bcf7 <unknown>
#5 0x5685c535ab9c <unknown>
#6 0x5685c534e4d6 <unknown>
#7 0x5685c537e492 <unknown>
#8 0x5685c534de89 <unknown>
#9 0x5685c537e65e <unknown>
#10 0x5685c539c9c6 <unkn